In [ ]:
import importlib
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import prism
import imagematerials.buildings.constants as bld_const
import imagematerials.buildings.preprocessing.circular_economy_measures as ce_mod
import imagematerials.buildings.preprocessing.main as bld_main
from imagematerials.util import read_circular_economy_config, read_climate_policy_config
import warnings
import prism

In [ ]:
# Force reload so the kernel picks up latest code
importlib.reload(bld_const)
importlib.reload(ce_mod)
importlib.reload(bld_main)
from imagematerials.buildings.preprocessing.main import buildings_preprocessing

base_dir = Path("..") / "data" / "raw"
climate_dir = base_dir / "image" / "SSP2_baseline"
ce_dirs = {
    "base": base_dir / "circular_economy_scenarios" / "base",
    "narrow_activity": base_dir / "circular_economy_scenarios" / "narrow_activity",
}

climate_cfg = read_climate_policy_config(climate_dir)
ce_cfg = read_circular_economy_config(ce_dirs)

comm_types = ["Office", "Retail+", "Hotels+", "Govt+"]
results = {}

with warnings.catch_warnings():
    warnings.filterwarnings("ignore")
    for mode in ("config", "convergence"):
        print(f"--- Running mode: {mode} ---")
        prep = buildings_preprocessing(base_dir, climate_cfg, ce_cfg,
                                       commercial_ce_mode=mode)
        
        stocks = prep["stocks"]
        stocks_comm = stocks.sel(Type=comm_types).sum("Type")

        n_nan = int(np.isnan(stocks_comm.values).sum())
        n_inf = int(np.isinf(stocks_comm.values).sum())
        total = float(np.nansum(stocks_comm.values[np.isfinite(stocks_comm.values)]))
        print(f"  NaN: {n_nan}, inf: {n_inf}, finite-sum: {total:.2e}")

        results[mode] = stocks_comm

print(f"\nConfig vs convergence differ? {not results['config'].equals(results['convergence'])}")

In [ ]:
# --- Plot 1: Global total commercial floorspace (all regions summed) ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for mode, stocks in results.items():
    total = stocks.sum("Region").loc[2000:]
    if prism.U_(total) is not None:
        total = total.pint.to("m**2")
    total.plot(ax=axes[0], label=mode)
axes[0].set_title("Global commercial floorspace stock (all regions)")
axes[0].set_ylabel("m²")
axes[0].legend()

# --- Plot 2: Per-region comparison for a few selected regions ---
sample_regions = ["CHN", "USA", "IND", "WEU"]
available = [r for r in sample_regions if r in results["config"].coords["Region"].values]

for mode, stocks in results.items():
    for reg in available:
        series = stocks.sel(Region=reg).loc[2000:]
        if prism.U_(series) is not None:
            series = series.pint.to("m**2")
        linestyle = "-" if mode == "convergence" else "--"
        series.plot(ax=axes[1], label=f"{reg} ({mode})", linestyle=linestyle)
axes[1].set_title("Commercial floorspace stock – selected regions")
axes[1].set_ylabel("m²")
axes[1].legend(fontsize=8, ncol=2)

plt.tight_layout()
plt.show()